<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-02-model-architecture/lesson-2.5-pre-training-at-scale/notebooks/exercises-2_5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 2.5: Pre-Training at Scale

*Module 2 · Large Language Models — Deep Dive*

> Training GPT-3 cost $4.6 million. LLaMA-3 used 15 trillion tokens across 16,000 GPUs for 54 days. This lesson covers the complete pre-training pipeline: data curation, distributed training with FSDP and DeepSpeed, Vertex AI multi-GPU jobs, and Chinchilla scaling laws.

`Data Curation` · `FSDP & DeepSpeed` · `Vertex AI` · `60 min`

---

## About this notebook

This notebook contains the **11 runnable code examples** from the published lesson `lesson-2.5.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Data Curation — Garbage In, Garbage Out — `01_data_pipeline.py`
2. Step 2 — The Training Loop — Next-Token Prediction at Scale — `02_training_loop.py`
3. Step 3 — Distributed Training — Why One GPU Isn't Enough — `03_fsdp_config.py`
4. Step 4 — Training on Vertex AI — Multi-GPU on Google Cloud — `04_vertex_training.py`
5. Step 5 — Scaling Laws — How Big Should the Model Be? — `05_scaling_calculator.py`
6. Step 7 — The Open Data Landscape — What Models Actually Train On — `data_landscape.py`
7. Step 8 — Training Cost Calculator — How Much Does Pre-Training Really Cost? — `cost_calculator.py`
8. Step 9 — Data Mixing, Curriculum & Monitoring — The Art of Pre-Training — `training_monitoring.py`
9. Step 10 — Training Instabilities & Flash Attention — What Goes Wrong at Scale — `instabilities.py`
10. Step 11 — Continual Pre-Training — Adapt Models Without Starting Over — `continual_pretraining.py`
11. Step 12 — Evaluation During Training & Open Science — `evaluation.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q torch datasets


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Data Curation — Garbage In, Garbage Out

~30-50% of crawled data survives cleaning. Quality filtering, PII removal, deduplication.

**`01_data_pipeline.py`** · _Block 1 of 11_

DATA CURATION PIPELINE — Clean web data for LLM training


In [ ]:
# DATA CURATION PIPELINE — Clean web data for LLM training
import re
from hashlib import md5

class DataPipeline:
    def filter_quality(self, text):
        if len(text.split()) < 20: return False  # too short
        if sum(1 for c in text if not c.isalnum() and c!=' ')/len(text) > 0.3: return False  # too many special chars
        words = text.lower().split()
        if len(set(words))/len(words) < 0.3: return False  # repetitive
        return True

    def remove_pii(self, text):
        text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL]', text)
        text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE]', text)
        return text

    def deduplicate(self, docs):
        seen, unique = set(), []
        for d in docs:
            h = md5(d.encode()).hexdigest()
            if h not in seen: seen.add(h); unique.append(d)
        return unique

    def process(self, docs):
        filtered = [d for d in docs if self.filter_quality(d)]
        cleaned = [self.remove_pii(d) for d in filtered]
        return self.deduplicate(cleaned)

# Simulate crawled data
raw = [
    "Machine learning is transforming how businesses operate across India.",
    "Buy now!!! Click here!!! $$$",
    "The the the the the the the",
    "Contact [email protected] or call 9876543210 for consulting.",
    "Machine learning is transforming how businesses operate across India.",  # dup
    "Python is the most popular language for data science and AI.",
    "hi",
    "Deep learning requires massive data and compute resources for training.",
]
clean = DataPipeline().process(raw)
print(f"Input: {len(raw)} docs → Output: {len(clean)} docs ({len(clean)/len(raw)*100:.0f}% survived)")


> **What just happened?** Quality filter removed spam and fragments. PII scrubber replaced emails/phones. Dedup removed copies. Only 50% survived — real pipelines for LLMs filter Common Crawl from 250B pages down to ~15T tokens. Data quality is the #1 determinant of model quality.


### Step 2 · The Training Loop — Next-Token Prediction at Scale

Predict next token → measure error → backprop → update → repeat trillions of times.

**`02_training_loop.py`** · _Block 2 of 11_

PRE-TRAINING LOOP — The core of every LLM


In [ ]:
# PRE-TRAINING LOOP — The core of every LLM
import torch, torch.nn as nn

class TinyLM(nn.Module):
    def __init__(self, vocab=1000, d=64, layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.enc = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=256, batch_first=True), layers)
        self.head = nn.Linear(d, vocab)
    def forward(self, x):
        mask = nn.Transformer.generate_square_subsequent_mask(x.size(1))
        return self.head(self.enc(self.embed(x), mask=mask, is_causal=True))

model = TinyLM()
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()
data = torch.randint(0, 1000, (100, 32))

print(f"Model: {sum(p.numel() for p in model.parameters()):,} params\n")
for epoch in range(5):
    total = 0
    for i in range(0, len(data), 10):
        b = data[i:i+10]
        loss = loss_fn(model(b[:,:-1]).reshape(-1,1000), b[:,1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"  Epoch {epoch+1}: loss={total/10:.3f}")
print(f"\nSame loop. GPT-3 = 175B params × 300B tokens. We used 700K × 3200.")


### Step 3 · Distributed Training — Why One GPU Isn't Enough

GPT-3 = 700GB in float32. One A100 = 80GB. You need multiple GPUs.

**`03_fsdp_config.py`** · _Block 3 of 11_

FSDP — Distributed Training with PyTorch


In [ ]:
# FSDP — Distributed Training with PyTorch
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy, MixedPrecision
import torch

fsdp_config = {
    "sharding_strategy": ShardingStrategy.FULL_SHARD,  # shard everything = ZeRO-3
    "mixed_precision": MixedPrecision(
        param_dtype=torch.bfloat16, reduce_dtype=torch.bfloat16, buffer_dtype=torch.bfloat16,
    ),
}
print("FSDP: shard params+grads+optimizer across GPUs")
print("  7B model without FSDP: ~56 GB (doesn't fit A100 40GB)")
print("  7B model with FSDP 4 GPUs: ~14 GB each ✅")
print("  Usage: model = FSDP(model, **fsdp_config)")


### Step 4 · Training on Vertex AI — Multi-GPU on Google Cloud

Launch distributed training jobs on GCP.

**`04_vertex_training.py`** · _Block 4 of 11_

VERTEX AI TRAINING JOB — Multi-GPU on GCP


In [ ]:
# VERTEX AI TRAINING JOB — Multi-GPU on GCP
from google.cloud import aiplatform

aiplatform.init(project="your-project", location="us-central1")

job = aiplatform.CustomJob(
    display_name="llm-pretraining",
    worker_pool_specs=[{
        "machine_spec": {
            "machine_type": "a2-highgpu-4g",  # 4× A100
            "accelerator_type": "NVIDIA_TESLA_A100",
            "accelerator_count": 4,
        },
        "replica_count": 1,
        "container_spec": {
            "image_uri": "us-docker.pkg.dev/vertex-ai/training/pytorch-gpu.2-2.py310:latest",
            "command": ["torchrun"],
            "args": ["--nproc_per_node=4", "train.py", "--use_fsdp=true"],
        },
    }],
)
print("Config: 4× A100, PyTorch 2.2, FSDP, ~$14/hr")
print("Launch: job.run()  |  Monitor: Vertex AI Console")


### Step 5 · Scaling Laws — How Big Should the Model Be?

Chinchilla proved most models were undertrained for their size.

**`05_scaling_calculator.py`** · _Block 5 of 11_

SCALING LAWS CALCULATOR — Chinchilla optimal


In [ ]:
# SCALING LAWS CALCULATOR — Chinchilla optimal
def estimate(params_b):
    tokens = params_b * 1e9 * 20
    flops = 6 * params_b * 1e9 * tokens
    gpu_hrs = flops / (156e12 * 3600)  # A100 bf16 @ 50% util
    cost = gpu_hrs / 8 * 3.5  # $3.50/hr per A100
    days = gpu_hrs / 8 / 24
    return tokens, gpu_hrs, cost, days

print(f"{'Model':>8s} {'Tokens':>10s} {'GPU-hrs':>12s} {'Days(8GPU)':>12s} {'Cost($)':>12s}")
for p in [0.1, 1, 7, 70, 175]:
    t, h, c, d = estimate(p)
    print(f"  {p:>6.1f}B {t/1e12:>8.1f}T {h:>10,.0f} {d:>10,.0f} {c:>10,.0f}")
print(f"\n99% of companies should FINE-TUNE, not pre-train.")


### Step 7 · The Open Data Landscape — What Models Actually Train On

FineWeb, RedPajama, SlimPajama — the public datasets powering open-source LLMs.

**`data_landscape.py`** · _Block 6 of 11_

OPEN PRE-TRAINING DATASETS (2024-2025)


In [ ]:
# =============================================
# OPEN PRE-TRAINING DATASETS (2024-2025)
# =============================================

datasets = [
    ("Common Crawl",    "~250B pages", "Raw web crawl — starting point for everything"),
    ("FineWeb",         "15T tokens",  "HuggingFace — heavily filtered Common Crawl"),
    ("RedPajama v2",    "30T tokens",  "Together AI — largest public dataset"),
    ("SlimPajama",      "627B tokens", "Deduplicated RedPajama (2.5x smaller)"),
    ("The Pile",        "825B tokens", "EleutherAI — diverse 22-source mix"),
    ("Dolma",           "3T tokens",   "AI2 — used for OLMo (fully open)"),
    ("StarCoder Data",  "783B tokens", "BigCode — 86 programming languages"),
]

print(f"{'Dataset':<18} {'Size':>12} Description")
print("-" * 70)
for name, size, desc in datasets:
    print(f"  {name:<16} {size:>12} {desc}")

print("""
DATA MIXING — LLaMA 3's recipe:
  Web (FineWeb/CC):  ~80%   General knowledge
  Code (GitHub):     ~10%   Reasoning + coding ability
  Books:              ~4%   Long-range coherence
  Wikipedia:          ~3%   Factual accuracy
  ArXiv/Papers:       ~2%   Scientific reasoning
  StackExchange:      ~1%   Q&A format

Why mix matters:
  More code → better at reasoning (even non-code tasks!)
  More books → better long-form coherence
  Too much web → more hallucinations
  The MIX is a trade secret — labs spend months tuning it.
""")


> **What just happened?** Open-source datasets range from 627B to 30T tokens. FineWeb (HuggingFace) is the gold standard: 15T tokens of heavily filtered web data. Data mixing is critical — LLaMA 3's ~10% code improves reasoning on ALL tasks, not just coding. The mix ratio is often more important than dataset size.


### Step 8 · Training Cost Calculator — How Much Does Pre-Training Really Cost?

From $100 for a hobby model to $100M+ for frontier models.

**`cost_calculator.py`** · _Block 7 of 11_

PRE-TRAINING COST ESTIMATION


In [ ]:
# =============================================
# PRE-TRAINING COST ESTIMATION
# =============================================

def estimate_cost(params_B, tokens_T, gpu_type="A100"):
    """Rough cost estimate for pre-training."""
    # FLOPs ≈ 6 × params × tokens (Kaplan approximation)
    flops = 6 * params_B * 1e9 * tokens_T * 1e12
    
    gpu_specs = {
        "A100": {"tflops": 312, "$/hr": 2.21},  # bf16, GCP spot
        "H100": {"tflops": 990, "$/hr": 3.50},  # bf16, estimate
    }
    spec = gpu_specs[gpu_type]
    
    # MFU (Model FLOPs Utilization) ≈ 40-50% in practice
    effective_tflops = spec["tflops"] * 0.45 * 1e12
    gpu_seconds = flops / effective_tflops
    gpu_hours = gpu_seconds / 3600
    cost = gpu_hours * spec["$/hr"]
    
    return gpu_hours, cost

configs = [
    ("Hobby (GPT-2 size)",  0.124, 0.010),  # 124M, 10B tokens
    ("Small (Phi-3 Mini)",  3.8,   3.3),    # 3.8B, 3.3T tokens
    ("LLaMA 3 8B",          8.0,   15.0),   # 8B, 15T tokens
    ("LLaMA 3 70B",         70.0,  15.0),   # 70B, 15T tokens
    ("GPT-4 (estimated)",   200.0, 13.0),   # ~200B active, ~13T tokens
]

print(f"{'Config':<22} {'GPU-hrs':>12} {'A100 Cost':>12} {'H100 Cost':>12}")
print("-" * 62)
for name, p, t in configs:
    a_hrs, a_cost = estimate_cost(p, t, "A100")
    h_hrs, h_cost = estimate_cost(p, t, "H100")
    print(f"  {name:<20} {a_hrs:>10,.0f}h ${a_cost:>10,.0f} ${h_cost:>10,.0f}")

print("""
KEY INSIGHT FOR YOUR CAREER:
  99% of companies should NOT pre-train. Fine-tune instead.
  Pre-training LLaMA 3 8B: ~$300K on A100s
  Fine-tuning LLaMA 3 8B with LoRA: ~$10 on Colab
  
  Pre-train ONLY if: you need a new language (IndicBERT), 
  a new domain (BloombergGPT), or a new modality (vision).
  
  Mixed-precision (bf16) cuts memory 2x with ~0% quality loss.
  Gradient checkpointing trades compute for memory (saves ~60%).
  These are essential — without them, training doesn't fit in GPU.
""")


> **What just happened?** Pre-training costs range from $100 (hobby GPT-2) to $100M+ (frontier). The 6NP rule: total FLOPs ≈ 6 × params × tokens. At 45% GPU utilization on A100s, LLaMA 3 8B costs ~$300K. For 99% of production use cases, fine-tuning ($10) beats pre-training ($300K). Mixed-precision (bf16) and gradient checkpointing are essential optimizations.


### Step 9 · Data Mixing, Curriculum & Monitoring — The Art of Pre-Training

How labs decide what data to feed, in what order, and when to stop.

**`training_monitoring.py`** · _Block 8 of 11_

TRAINING MONITORING — What to Watch


In [ ]:
# =============================================
# TRAINING MONITORING — What to Watch
# =============================================

print("""
LOSS CURVE — The #1 Metric:
  Healthy:     Smooth downward curve, ~3.0 → ~2.5 over training
  Loss spike:  Sudden jump → learning rate too high or bad data batch
  Plateau:     Flat curve → model capacity exhausted or lr too low
  Divergence:  Loss going UP → gradient explosion, restart needed

MONITORING CHECKLIST:
  ✅ Training loss (per step)
  ✅ Validation loss (every N steps) — gap = overfitting
  ✅ Gradient norm — should be stable, not spiking
  ✅ GPU utilization — should be >90%, else bottleneck
  ✅ Learning rate schedule — warmup then cosine decay
  ✅ Tokens/second throughput — measure efficiency

CURRICULUM LEARNING (2025 trend):
  Stage 1: General web data (breadth) — 80% of training
  Stage 2: High-quality curated data (depth) — 20% of training
  
  IBM's GneissWeb: "Train on 10T general tokens first,
  then refine on 0.5T high-quality tokens"
  
  This is like education: learn broadly first,
  then specialize. The final 20% matters most.

SYNTHETIC DATA (emerging trend):
  - Use existing LLMs to generate training data
  - Phi-3: trained significantly on GPT-4-generated data
  - DeepSeek: uses self-generated reasoning traces
  - Risk: "model collapse" if training only on synthetic data
  - Best practice: mix synthetic + real data
""")


> **What just happened?** Three production insights: (1) Loss curves tell you everything — spikes mean bad data or high lr, plateaus mean capacity exhaustion. (2) Curriculum learning (general→specialized) improves final quality — the last 20% of training on curated data matters most. (3) Synthetic data is the 2025 trend (Phi-3, DeepSeek) but must be mixed with real data to avoid model collapse.


### Step 10 · Training Instabilities & Flash Attention — What Goes Wrong at Scale

Loss spikes, NaN explosions, and the one optimization you can't skip.

**`instabilities.py`** · _Block 9 of 11_

TRAINING INSTABILITIES — The #1 operational headache


In [ ]:
# =============================================
# TRAINING INSTABILITIES — The #1 operational headache
# =============================================

print("""
WHAT GOES WRONG AT SCALE:
  Loss spike:       Sudden jump in loss → bad data batch or lr too high
  NaN explosion:    Gradients → infinity → model weights become NaN → training dead
  Gradient explosion: Residual connections amplify gradients across deep networks
  fp16 overflow:    Half-precision can't represent very large/small numbers
                    (Pythia 1B had to switch from fp16 → bf16 mid-training!)

OLMo 2's STABILITY STACK (what actually works):
  1. RMSNorm (not LayerNorm) — more stable gradient flow
  2. QK-Norm — normalize Q and K before attention (prevents attention explosions)
  3. Z-loss regularization — soft penalty on large logits
  4. Improved initialization — scale down residual layers by 1/√N
  → Each technique alone is insufficient. Together = no crashes.

RECOVERY STRATEGIES:
  - Checkpoint every N steps (save model + optimizer + scheduler state)
  - On spike: roll back to last good checkpoint, reduce lr by 50%
  - On NaN: likely irrecoverable — restart from checkpoint with bf16
  - Monitor gradient norm: should be stable 0.5-2.0, spike > 10 = trouble

FLASH ATTENTION — Not optional, it's infrastructure:
  Standard attention: O(N²) memory (sequence length²)
  Flash Attention:    O(N) memory via tiling + recomputation
  
  How it works:
    - Splits Q, K, V into tiles that fit in GPU SRAM (fast memory)
    - Computes attention tile-by-tile, never materializing N×N matrix
    - Recomputes attention during backward pass instead of storing
  
  Impact: 2-4x speedup, enables 4x longer context lengths
  Flash Attention 2: Better GPU parallelism (split across warps)
  Flash Attention 3: Hopper-specific (H100), further 1.5-2x speedup
  
  Without FlashAttention, you can't train with context > 2048 tokens
  on a single A100. With it, 8192+ tokens fit easily.
""")


> **What just happened?** Training instabilities are the #1 reason real pre-training runs fail. OLMo 2 solved this by stacking RMSNorm + QK-Norm + Z-loss + improved initialization — each alone wasn't enough. Flash Attention reduces attention memory from O(N²) to O(N) — it's not an optimization, it's infrastructure. Without it, long contexts don't fit in GPU memory. Always checkpoint regularly and monitor gradient norms.


### Step 11 · Continual Pre-Training — Adapt Models Without Starting Over

The most production-relevant pre-training skill. Cheaper than from-scratch, more powerful than fine-tuning.

**`continual_pretraining.py`** · _Block 10 of 11_

CONTINUAL PRE-TRAINING (CPT) — The sweet spot


In [ ]:
# =============================================
# CONTINUAL PRE-TRAINING (CPT) — The sweet spot
# =============================================

print("""
WHEN TO USE EACH APPROACH:
  From-scratch pre-training: New language, new modality, research
    Cost: $100K-100M+ | Time: weeks-months | Data: trillions of tokens

  Continual pre-training (CPT): New domain on existing model
    Cost: $1K-50K | Time: hours-days | Data: billions of domain tokens
    Example: LLaMA 3 8B → medical LLaMA (train on PubMed/clinical notes)

  Fine-tuning (SFT/LoRA): Task-specific adaptation
    Cost: $1-100 | Time: minutes-hours | Data: thousands of examples
    Example: LLaMA 3 8B → customer support chatbot

DECISION TREE:
  Is the domain very different from web text?
    YES → Continual pre-training first, THEN fine-tune
    NO  → Just fine-tune (Module 7)
  
  Do you need a new language (Hindi, Tamil, Telugu)?
    YES → CPT with Indic language corpus (IndicCorp, Sangraha)
    NO  → Fine-tune with multilingual model

CRITICAL CPT RULES:
  1. NEVER CPT an instruction-tuned model → loses instruction ability!
     Always CPT the BASE model, then instruction-tune after
  2. Use MUCH lower learning rate than initial pre-training
     Initial: 3e-4 → CPT: 1e-5 to 5e-5 (10-30x lower)
  3. Mix domain data with general data (80/20) to prevent forgetting
  4. WSD scheduling is ideal — stable phase extends indefinitely
  
REAL EXAMPLE — Databricks research:
  CPT on LLaMA-2-7B → matched LLaMA-2-13B on MMLU
  That's getting 13B performance from a 7B model via domain data!
  
FOR INDIAN ENGINEERS:
  - Medical: CPT on Indian clinical notes + PubMed
  - Legal: CPT on Indian case law + statutes  
  - Financial: CPT on SEBI filings + RBI reports
  - Indic languages: CPT on IndicCorp (AI4Bharat, 22 languages)
""")


> **What just happened?** Continual pre-training (CPT) is the most production-relevant pre-training skill. It's 100-1000x cheaper than from-scratch pre-training and gives deeper domain knowledge than fine-tuning. Key rules: always CPT the BASE model (never instruction-tuned), use 10-30x lower learning rate, mix domain+general data. Databricks showed CPT can make a 7B model match a 13B model. For Indian domains: medical, legal, financial, and Indic language CPT on open corpora.


### Step 12 · Evaluation During Training & Open Science

How do you know if training is working? Benchmarks, perplexity, and the contamination problem.

**`evaluation.py`** · _Block 11 of 11_

EVALUATION DURING TRAINING


In [ ]:
# =============================================
# EVALUATION DURING TRAINING
# =============================================
import math

# Perplexity — the standard pre-training metric
def perplexity(avg_loss):
    """Lower perplexity = model is more confident = better."""
    return math.exp(avg_loss)

# Example: tracking improvement
losses = [4.5, 3.8, 3.2, 2.8, 2.5]  # loss at each epoch
for i, loss in enumerate(losses):
    ppl = perplexity(loss)
    print(f"  Epoch {i+1}: loss={loss:.1f} → perplexity={ppl:.0f}")

print("""
BENCHMARK EVALUATION (run periodically during training):
  MMLU:      Multiple-choice across 57 subjects (world knowledge)
  HellaSwag: Commonsense reasoning (sentence completion)
  ARC:       Science questions (reasoning)
  GSM8K:     Grade-school math (arithmetic reasoning)
  HumanEval: Code generation (pass@1)
  
  OLMo uses OLMES with 20 benchmarks for maximum signal.
  Karpathy's nanochat evaluates on 22 datasets → single CORE score.

PERPLEXITY'S LIMITATION (2025 research):
  Perplexity does NOT reliably predict downstream performance!
  Two models with identical perplexity can differ by 10%+ on MMLU.
  → Use perplexity for training monitoring, benchmarks for selection.

DATA CONTAMINATION — The evaluation crisis:
  If benchmark questions leaked into training data → inflated scores
  OLMo 3 introduced OlmoTrace: traces outputs to training data
  Best practice: hold out eval data, use multiple benchmarks

OPEN SCIENCE EXEMPLARS (learn from their failures):
  OLMo (AI2):  Fully open — data, code, weights, training logs
               OLMo 2: RMSNorm+QK-Norm fixed instability after OLMo 1 failed
               OLMo 3: 2.5x more efficient than LLaMA 3.1 in GPU-hours/token
  
  Pythia (EleutherAI): 16 models × 154 checkpoints each
               Proved: different random seeds → meaningfully different models
               Training is inherently non-deterministic!
""")


> **What just happened?** Perplexity = exp(loss) is the standard training metric, but 2025 research shows it doesn't reliably predict downstream performance — always complement with benchmarks. Data contamination inflates scores; OLMo 3's OlmoTrace links outputs to training data for auditing. Open science projects like OLMo and Pythia document real failures and recovery — invaluable for understanding what actually happens at scale.


---

## Wrap-up

You've walked through all 11 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
